# 0. Import library and load data

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
sns.set(style="whitegrid")
import re

In [2]:
df = pd.read_json("../../Data/Shopee/Cosmetic_Shopee_Data.json")

# 1. Data dropping

As we mention in the data understanding progress, we need to drop two of the attribute columns:
- ``Platform``: There is only one unique value which is "Shopee" indicates the only platform we crawl. Therefore, we will drop this meaningless attribute while putting this information as the name of database
- ``Brand``: Most of the value are "No Brand" which mean null value returned by API according to the crawling code

In [3]:
df_clean = df.copy()
df_clean = df_clean.drop(columns = ['Platform', 'Brand'])

# 2. Data type conversion

As we have already dropped two columns, the dataset now have 45103 rows with only 19 columns left. Here is the detailed information along with data type of every attribute:

In [4]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45104 entries, 0 to 45103
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Main_Category     45104 non-null  object 
 1   Sub_Category      45104 non-null  object 
 2   ID                45104 non-null  int64  
 3   Shop_ID           45104 non-null  int64  
 4   Name              45104 non-null  object 
 5   Price             45104 non-null  int64  
 6   Original_Price    45104 non-null  int64  
 7   Discount_Percent  45104 non-null  float64
 8   Rating            45104 non-null  float64
 9   Reviews           45104 non-null  int64  
 10  Monthly_Sold      45104 non-null  int64  
 11  Total_Sold        45104 non-null  int64  
 12  Liked_Count       45104 non-null  int64  
 13  Shop_Location     45104 non-null  object 
 14  Is_Mall           45104 non-null  int64  
 15  Is_Verified       45104 non-null  int64  
 16  Is_FreeShip       45104 non-null  int64 

However, as the result above, there are some invalid data type that we need to convert:

| Column               | Expected Dtype | Description                                  |
| :------------------- | :------------- | :------------------------------------------- |
| **ID**               | string         | Unique product identifier                    |
| **Shop_ID**          | string         | Unique shop identifier                       |
| **Name**             | object         | Product name/title                           |
| **Main_Category**    | category       | Main product category                        |
| **Sub_Category**     | category       | Detailed product category                    |
| **Price**            | int            | Current selling price (VND)                  |
| **Original_Price**   | int            | Original price before discount               |
| **Discount_Percent** | float          | Discount percentage                          |
| **Rating**           | float          | Average rating (0–5)                         |
| **Reviews**          | int            | Number of reviews                            |
| **Monthly_Sold**     | int            | Monthly units sold                           |
| **Total_Sold**       | int            | Total units sold                             |
| **Liked_Count**      | int            | Number of likes                              |
| **Shop_Location**    | category       | Shop location                                |
| **Is_Mall**          | bool           | Shopee Mall indicator                        |
| **Is_Verified**      | bool           | Verified shop indicator                      |
| **Is_FreeShip**      | bool           | Free shipping indicator                      |
| **Is_Ads**           | bool           | Advertisement indicator                      |
| **Created_At_Days**  | int            | Product age (days since listing)             |

- `ID`, `Shop_ID` are converted to **string** to avoid numeric misinterpretation
- **category** is used for low-cardinality categorical variables to reduce memory
- Binary features are converted to **boolean**

In [5]:
# Convert identifier columns
df_clean['ID'] = df_clean['ID'].astype(str)
df_clean['Shop_ID'] = df_clean['Shop_ID'].astype(str)

In [6]:
# Convert categorical columns
categorical_cols = ['Main_Category', 'Sub_Category', 'Shop_Location']
df_clean[categorical_cols] = df_clean[categorical_cols].astype('category')

In [7]:
# Convert binary columns
binary_cols = ['Is_Mall', 'Is_Verified', 'Is_FreeShip', 'Is_Ads']
df_clean[binary_cols] = df_clean[binary_cols].astype('bool')

In [8]:
# Convert numerical columns
int_cols = [
    'Price', 'Original_Price', 'Reviews',
    'Monthly_Sold', 'Total_Sold', 'Liked_Count'
]
df_clean[int_cols] = df_clean[int_cols].apply(pd.to_numeric, downcast='integer')

float_cols = ['Discount_Percent', 'Rating', 'Created_At_Days']
df_clean[float_cols] = df_clean[float_cols].apply(pd.to_numeric, downcast='float')

In [9]:
# Convert Created_At_Days to integer
df_clean['Created_At_Days'] = df_clean['Created_At_Days'].astype('int32')

Finally, we double check our conversion result by using ``info()`` method once again:

In [10]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45104 entries, 0 to 45103
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   Main_Category     45104 non-null  category
 1   Sub_Category      45104 non-null  category
 2   ID                45104 non-null  object  
 3   Shop_ID           45104 non-null  object  
 4   Name              45104 non-null  object  
 5   Price             45104 non-null  int32   
 6   Original_Price    45104 non-null  int32   
 7   Discount_Percent  45104 non-null  float32 
 8   Rating            45104 non-null  float32 
 9   Reviews           45104 non-null  int32   
 10  Monthly_Sold      45104 non-null  int32   
 11  Total_Sold        45104 non-null  int32   
 12  Liked_Count       45104 non-null  int32   
 13  Shop_Location     45104 non-null  category
 14  Is_Mall           45104 non-null  bool    
 15  Is_Verified       45104 non-null  bool    
 16  Is_FreeShip       4510

The result is perfectly fine and the memory usage reduce 3 times by using the suitable data types for the entire dataset

# 3. Data cleaning

## 3.1 Attribute ``Main_Category``

We need to translate the value to Vietnamese for dashboard

In [11]:
main_category_map = {
    'Skin Care': 'Chăm sóc da',
    'Hair Care': 'Chăm sóc tóc',
    'Makeup': 'Trang điểm',
    'Beauty_Devices': 'Thiết bị làm đẹp',
    'Body Care': 'Chăm sóc cơ thể',
    'Oral Care': 'Chăm sóc răng miệng',
    'Fragrance': 'Nước hoa',
    'Dermocosmetic': 'Dược mỹ phẩm'
}

df_clean['Main_Category'] = df_clean['Main_Category'].map(main_category_map)
df_clean['Main_Category'] = df_clean['Main_Category'].astype('category')

In [12]:
df_clean['Main_Category'].unique()

['Thiết bị làm đẹp', 'Chăm sóc cơ thể', 'Dược mỹ phẩm', 'Nước hoa', 'Chăm sóc tóc', 'Trang điểm', 'Chăm sóc răng miệng', 'Chăm sóc da']
Categories (8, object): ['Thiết bị làm đẹp', 'Chăm sóc cơ thể', 'Dược mỹ phẩm', 'Nước hoa', 'Chăm sóc tóc', 'Trang điểm', 'Chăm sóc răng miệng', 'Chăm sóc da']

## 3.2 Attribute ``Sub_Category``

Most of the value from ``Sub_Category`` is fully in lower-case mode. We need to use upper-case letter at the beginning of the word instead. 

In [13]:
df_clean['Sub_Category'] = (
    df_clean['Sub_Category']
    .astype(str)
    .str.strip()
    .str.title()
)
df_clean['Sub_Category'] = df_clean['Sub_Category'].astype('category')

In [14]:
df_clean['Sub_Category'].unique()

['Lược Điện', 'Máy Massage Mặt', 'Máy Rửa Mặt', 'Máy Sấy Tóc', 'Máy Triệt Lông', ..., 'Serum Dưỡng Da', 'Sữa Rửa Mặt', 'Tẩy Trang', 'Tẩy Tế Bào Chết Da Mặt', 'Xịt Khoáng']
Length: 63
Categories (63, object): ['Aha', 'Azelaic Acid', 'Bha', 'Body Lotion', ..., 'Tẩy Tế Bào Chết Body', 'Tẩy Tế Bào Chết Da Mặt', 'Xịt Dưỡng Tóc', 'Xịt Khoáng']

## 3.3 Attribute ``Rating``

We need to round the rating floating value to 2 decimal places. The current value is to detailed tp to 7 decimal places which will be unecessary

In [15]:
df_clean['Rating'] = df_clean['Rating'].round(2)

In [16]:
df_clean['Rating'].unique()

array([0.  , 4.99, 4.75, 4.84, 5.  , 4.85, 4.6 , 4.83, 4.87, 4.95, 3.33,
       4.88, 4.82, 4.81, 4.72, 4.86, 4.94, 4.97, 3.67, 4.92, 4.93, 4.4 ,
       4.67, 4.96, 4.9 , 4.79, 4.91, 4.45, 4.66, 4.62, 4.73, 4.98, 4.44,
       4.43, 4.8 , 4.7 , 4.76, 4.74, 4.64, 4.78, 4.69, 4.89, 4.39, 3.86,
       4.63, 4.71, 3.  , 4.  , 4.23, 2.  , 4.58, 2.75, 4.2 , 4.5 , 4.65,
       3.77, 4.3 , 4.32, 4.36, 3.54, 4.25, 4.59, 4.54, 4.61, 4.77, 4.41,
       4.56, 4.68, 4.47, 3.85, 4.42, 3.64, 4.33, 4.51, 3.5 , 4.52, 1.5 ,
       4.57, 3.25, 4.49, 4.38, 3.4 , 4.27, 4.55, 4.48, 4.46, 4.29, 4.53,
       1.  , 4.31, 3.75, 2.5 , 4.35, 4.17, 4.24, 2.33, 3.97, 4.19, 1.6 ,
       4.11, 3.62, 4.21, 4.1 , 4.08, 3.83, 3.11, 2.84, 3.43, 4.22, 3.07,
       2.65, 3.6 , 2.47, 3.89, 3.03, 4.12, 3.8 , 3.58, 3.1 , 4.07, 3.47,
       3.38, 3.88, 3.93, 4.09, 3.24, 3.31, 3.29, 3.66, 4.05, 2.96, 3.57,
       3.46, 3.04, 4.37, 4.14, 3.7 , 3.9 , 4.16, 3.2 , 3.94, 1.33, 3.69,
       4.34, 2.6 , 3.92, 2.83, 2.44, 3.36, 4.28, 2.

## 3.4 Attribute ``Shop_Location``

We first need to drop the row whose shop location's value is invalid. Those values are empty strings which do not provide any meaningful information

In [17]:
df_clean['Shop_Location'] = df_clean['Shop_Location'].astype(str).str.strip()
df_clean = df_clean[df_clean['Shop_Location'] != '']

There also inconsistent value of foreign location. We use the term "Nước ngoài" to represent all of the foreign countries 

In [18]:
foreign_map = {
    'Nhật Bản': 'Nước ngoài',
    'Hàn Quốc': 'Nước ngoài',
    'Trung Quốc': 'Nước ngoài',
    'Hong Kong': 'Nước ngoài'
}

df_clean['Shop_Location'] = df_clean['Shop_Location'].replace(foreign_map)
df_clean['Shop_Location'] = df_clean['Shop_Location'].astype('category')

We also need to convert city to the consistent format. For example, "Hà Nội" to "Thành phố Hà Nội" or "TP. Hồ Chí Minh" to "Thành phố Hồ Chí Minh". Here is the plan:
- **Step 1**: Convert "TP." to "Thành phố"
- **Step 2**: Add "Thành phố" prefix to city
- **Step 3**: Add "Tỉnh" prefix to the remaining provinces

In [19]:
df_clean['Shop_Location'] = (
    df_clean['Shop_Location']
    .str.replace(r'\bTP\.?\s*', 'Thành phố ', regex=True)
)

In [20]:
city_map = {
    'Hà Nội': 'Thành phố Hà Nội',
    'Đà Nẵng': 'Thành phố Đà Nẵng',
    'Cần Thơ': 'Thành phố Cần Thơ',
    'Hải Phòng': 'Thành phố Hải Phòng',
    'Thừa Thiên Huế': "Thành phố Huế"
}

df_clean['Shop_Location'] = df_clean['Shop_Location'].replace(city_map)
df_clean['Shop_Location'] = df_clean['Shop_Location'].astype('category')

In [21]:
def location_prefix(x):
    if x.startswith('Thành phố'):
        return x
    elif x.startswith('Tỉnh'):
        return x
    elif x == 'Nước ngoài':
        return x
    else:
        return f'Tỉnh {x}'

df_clean['Shop_Location'] = df_clean['Shop_Location'].apply(location_prefix)

Vietnam has undergone administrative changes, reducing the number of provinces and cities through mergers and boundary adjustments.

To ensure consistency with the current administrative system, the `Shop_Location` column is transformed by mapping outdated province names to their updated equivalents.

In [22]:
province_group_map = {
    'Tỉnh Tuyên Quang': 'Tỉnh Tuyên Quang',
    'Tỉnh Hà Giang': 'Tỉnh Tuyên Quang',

    'Tỉnh Lào Cai': 'Tỉnh Lào Cai',
    'Tỉnh Yên Bái': 'Tỉnh Lào Cai',

    'Tỉnh Thái Nguyên': 'Tỉnh Thái Nguyên',
    'Tỉnh Bắc Kạn': 'Tỉnh Thái Nguyên',

    'Tỉnh Vĩnh Phúc': 'Tỉnh Phú Thọ',
    'Tỉnh Phú Thọ': 'Tỉnh Phú Thọ',
    'Tỉnh Hòa Bình': 'Tỉnh Phú Thọ',

    'Tỉnh Bắc Ninh': 'Tỉnh Bắc Ninh',
    'Tỉnh Bắc Giang': 'Tỉnh Bắc Ninh',

    'Tỉnh Hưng Yên': 'Tỉnh Hưng Yên',
    'Tỉnh Thái Bình': 'Tỉnh Thái Bình',

    'Tỉnh Hải Dương': 'Thành phố Hải Phòng',
    'Thành phố Hải Phòng': 'Thành phố Hải Phòng',

    'Tỉnh Ninh Bình': 'Tỉnh Ninh Bình',
    'Tỉnh Nam Định': 'Tỉnh Ninh Bình',
    'Tỉnh Hà Nam': 'Tỉnh Ninh Bình',

    'Tỉnh Quảng Trị': 'Tỉnh Quảng Trị',
    'Tỉnh Quảng Bình': 'Tỉnh Quảng Trị',

    'Thành phố Đà Nẵng': 'Thành phố Đà Nẵng',
    'Tỉnh Quảng Nam': 'Thành phố Đà Nẵng',

    'Tỉnh Quảng Ngãi': 'Tỉnh Quảng Ngãi',
    'Tỉnh Kon Tum': 'Tỉnh Quảng Ngãi',

    'Tỉnh Gia Lai': 'Tỉnh Gia Lai',
    'Tỉnh Bình Định': 'Tỉnh Gia Lai',

    'Tỉnh Khánh Hòa': 'Tỉnh Khánh Hòa',
    'Tỉnh Ninh Thuận': 'Tỉnh Khánh Hòa',

    'Tỉnh Lâm Đồng': 'Tỉnh Lâm Đồng', 
    'Tỉnh Bình Thuận': 'Tỉnh Lâm Đồng',
    'Tỉnh Đắk Nông': 'Tỉnh Lâm Đồng',

    'Tỉnh Đắk Lắk': 'Tỉnh Đắk Lắk',
    'Tỉnh Phú Yên': 'Tỉnh Đắk Lắk',

    'Thành phố Hồ Chí Minh': 'Thành phố Hồ Chí Minh',
    'Tỉnh Bình Dương': 'Thành phố Hồ Chí Minh',
    'Tỉnh Bà Rịa - Vũng Tàu': 'Thành phố Hồ Chí Minh',

    'Tỉnh Đồng Nai': 'Tỉnh Đồng Nai',
    'Tỉnh Bình Phước': 'Tỉnh Đồng Nai',

    'Tỉnh Tây Ninh': 'Tỉnh Tây Ninh',
    'Tỉnh Long An': 'Tỉnh Tây Ninh',

    'Thành phố Cần Thơ': 'Thành phố Cần Thơ',
    'Tỉnh Sóc Trăng': 'Thành phố Cần Thơ', 
    'Tỉnh Hậu Giang': 'Thành phố Cần Thơ',

    'Tỉnh Vĩnh Long': 'Tỉnh Vĩnh Long',
    'Tỉnh Bến Tre': 'Tỉnh Vĩnh Long',
    'Tỉnh Trà Vinh': 'Tỉnh Vĩnh Long',

    'Tỉnh Đồng Tháp': 'Tỉnh Đồng Tháp',
    'Tỉnh Tiền Giang': 'Tỉnh Đồng Tháp',

    'Tỉnh Cà Mau': 'Tỉnh Cà Mau',
    'Tỉnh Bạc Liêu': 'Tỉnh Cà Mau',

    'Tỉnh An Giang': 'Tỉnh An Giang',
    'Tỉnh Kiên Giang': 'Tỉnh An Giang'
}

# Apply mapping
df_clean['Shop_Location'] = df_clean['Shop_Location'].replace(province_group_map)
df_clean['Shop_Location'] = df_clean['Shop_Location'].astype('category')

In [23]:
pd.set_option('display.max_rows', None)
print(df_clean["Shop_Location"].value_counts())

Shop_Location
Thành phố Hồ Chí Minh    16696
Thành phố Hà Nội         15811
Nước ngoài                4505
Tỉnh Bắc Ninh             1378
Thành phố Đà Nẵng          743
Tỉnh Tây Ninh              691
Tỉnh Ninh Bình             542
Tỉnh Thanh Hóa             468
Tỉnh Thái Nguyên           447
Thành phố Hải Phòng        418
Tỉnh Đồng Nai              406
Thành phố Cần Thơ          388
Tỉnh Hưng Yên              277
Tỉnh Phú Thọ               269
Tỉnh Đồng Tháp             208
Tỉnh Vĩnh Long             192
Tỉnh An Giang              154
Tỉnh Lạng Sơn              153
Tỉnh Đắk Lắk               137
Tỉnh Lâm Đồng              130
Tỉnh Quảng Ninh            127
Tỉnh Cà Mau                123
Thành phố Huế              117
Tỉnh Thái Bình             117
Tỉnh Gia Lai               100
Tỉnh Khánh Hòa              88
Tỉnh Lào Cai                64
Tỉnh Nghệ An                63
Tỉnh Quảng Trị              44
Tỉnh Quảng Ngãi             26
Tỉnh Hà Tĩnh                24
Tỉnh Sơn La              

# 4. Vietnamese translation

In [24]:
df_clean = df_clean.rename(columns={
    'Main_Category': 'Danh_muc_chinh',
    'Sub_Category': 'Danh_muc_phu',
    'ID': 'Ma_san_pham',
    'Shop_ID': 'Ma_cua_hang',
    'Name': 'Ten_san_pham',
    'Price': 'Gia',
    'Original_Price': 'Gia_goc',
    'Discount_Percent': 'Phan_tram_giam_gia',
    'Rating': 'Danh_gia',
    'Reviews': 'So_danh_gia',
    'Monthly_Sold': 'Doanh_so_thang',
    'Total_Sold': 'Doanh_so_tong',
    'Liked_Count': 'Luot_thich',
    'Shop_Location': 'Dia_diem_cua_hang',
    'Is_Mall': 'La_mall',
    'Is_Verified': 'Da_xac_thuc',
    'Is_FreeShip': 'Mien_phi_van_chuyen',
    'Is_Ads': 'Co_quang_cao',
    'Created_At_Days': 'So_ngay_da_ban'
})

# 5. Final cleaned dataset

In [25]:
df_clean.head()

,Danh_muc_chinh,Danh_muc_phu,Ma_san_pham,Ma_cua_hang,Ten_san_pham,Gia,Gia_goc,Phan_tram_giam_gia,Danh_gia,So_danh_gia,Doanh_so_thang,Doanh_so_tong,Luot_thich,Dia_diem_cua_hang,La_mall,Da_xac_thuc,Mien_phi_van_chuyen,Co_quang_cao,So_ngay_da_ban
0,Thiết bị làm đẹp,Lược Điện,42327913165,1681939005,Phong Cách Mới Sạc Không Dây Di Động LCD Tóc T...,139523,233123,40.200001,0.00,0,1,3,0,Nước ngoài,False,False,False,False,85
1,Thiết bị làm đẹp,Lược Điện,42417976686,1620193358,Lược Điện Chải Tóc Không Dây Ion Âm JC-888 See...,183000,350000,47.700001,4.99,270,2000,2000,4,Thành phố Hồ Chí Minh,False,False,False,False,210
2,Thiết bị làm đẹp,Lược Điện,47357926177,1506174776,Lược Điện Uốn Mái Mini KOIZUMI Không Dây Tạo K...,908500,1485000,38.799999,4.75,52,1,463,2462,Thành phố Hà Nội,False,True,False,False,180
3,Thiết bị làm đẹp,Lược Điện,51102137858,1634218010,Lược Điện Chải Tóc Ion Âm Lược Duỗi Tóc Không...,195000,250000,22.000000,4.84,171,1000,1000,23,Thành phố Hà Nội,False,False,False,False,131
4,Thiết bị làm đẹp,Lược Điện,55400131427,1252295440,Lược Điện Không Dây SeeMee Chăm Sóc Tóc Suôn M...,189000,509000,62.900002,5.00,189,1000,1000,27,Thành phố Hồ Chí Minh,False,True,False,False,163


In [26]:
df_clean.to_json(
    '../../Data/Shopee/cleaned_shopee_data.json',
    orient='records',
    double_precision=2,
    force_ascii=False,
    indent=4
)